# Project notebook with R
- Easy access to all the experimental data, statistical analysis results and visualization of reports using R
- All the R functionality at hand

## Import R libraries
Make sure that all the libraries you plan to use have been previously installed.

If not, you can install them through the notebook.

E.g. *install.packages("tidyverse")*

In [ ]:
library(rhdf5)
library(rjson)
library(foreach)
library(plotly)
library(tidyverse)
library(networkD3)

## Define custom functions 
Within the Jupyter notebook you can create all the functions relevant to you work that are not implemented in the analytics core. This is the case of R functions. Here we define R functions that allow us to load a project from the CKG database, access different datasets and plots, and visualize the latter in the notebook.

In [ ]:
importGraph <- function(json_network){
  edges_df = data.frame(matrix(unlist(json_network$net_json$links), nrow=length(json_network$net_json$links), byrow=T))
  colnames(edges_df)<-names(unlist(json_network$net_json$links[[1]]))
  edges_df <- edges_df[c("source", "target", "width")]
  colnames(edges_df)<-c("source", "target", "weight")
  nodes = data.frame(matrix(unlist(json_network$net_json$nodes), nrow=length(json_network$net_json$nodes), byrow=T))
  colnames(nodes)<-names(unlist(json_network$net_json$nodes[[1]]))
  nodes['ident']<- 0:(dim(nodes)[1]-1)
  nodes <- nodes[c("ident", "id", "color")]
  colnames(nodes) <- c("id", "label", "color")
  edges <- edges_df %>% 
    left_join(nodes, by = c("source" = "label")) %>% 
    rename(from = id)
  edges <- edges %>% 
    left_join(nodes, by = c("target" = "label")) %>% 
    rename(to = id)
  edges <- edges[c("from", "to", "weight")]
  library(networkD3)
  g<-forceNetwork(Links = edges, Nodes = nodes,
                  height = 850, width = 800,
                  Source = "from", Target = "to",
                  Value = "weight", NodeID = "label",
                  Group = "color", opacity = 0.8,opacityNoHover = 1,
                  zoom = T, linkWidth = JS("function(d) { return Math.sqrt(d.weight); }"))
  return(g)
}

read_report <- function(report_file, report_name){
  plots <- c()
  nets <- c()
  if(file.exists(report_file)){
    content <- h5ls(report_file)
    foreach(group=unique(content$group)) %do% {
      if(group != "/"){
        if(!grepl('Table', group)){
          group = substring(group, 2)
          report_figure <- h5read(report_file, group)
          foreach(name=unique(names(report_figure))) %do% {
            if(grepl('figure',name)){
              report_figure_str<-fromJSON(report_figure[[name]][1])
              p = report_figure_str$props$figure
              plot <- list(plot=p)
              names(plot) <- paste(group,name,sep="_")
              plots<-append(plots, plot)
            }
            else if(grepl('net', name)){
              report_net_str <- fromJSON(report_figure[[name]][1])
              net <- list(net=report_net_str)
              names(net) <- name
              nets<-append(nets, net)
            }
          }
        }
      }
    }
  }
  h5closeAll()
  report <- list(name = report_name, plots = plots, nets = nets)
  class(report) <- "report"
  
  return(report)
}

read_dataset <- function(dataset_file, dataset_name){
  dataframes <- c()
  if(file.exists(dataset_file)){
    content <- h5ls(dataset_file)
    foreach(group=unique(content$group)) %do% {
      if(group != "/"){
        group = substring(group, 2)
        dataframe_json <- h5read(dataset_file, group)
        foreach(name=unique(names(dataframe_json))) %do% {
          if(is.null(names(dataframe_json[[name]]))){
          dataframe_str<-fromJSON(dataframe_json[[name]][1])}
          df <- data.frame(matrix(unlist(dataframe_str), nrow=length(dataframe_str), byrow=T))
          colnames(df)<-names(unlist(dataframe_str[[1]]))
          dataframe <- list(data=df)
          names(dataframe) <- name
          dataframes<-append(dataframes, dataframe)
        }
      }
    }
  }
  h5closeAll()
  dataset <- list(name=dataset_name, data=dataframes)
  class(dataset) <- "dataset"
  
  return(dataset)
}


load_project <- function(project_id, dataset){
  plots = c()
  nets = c()
  project_report_dir <- paste("../../../data/reports/", project_id, sep = "")
  project_dataset_report_dir <- paste(project_report_dir,dataset, sep="/")
  report_file = paste(project_dataset_report_dir, "report.h5", sep = "/")
  datasets_file = paste(project_dataset_report_dir, paste(dataset,"dataset.h5",sep='_'), sep = "/")
  report = read_report(report_file, paste(project_id, "Report", dataset))
  datasets = read_dataset(datasets_file, paste(project_id, "Dataset", dataset))
  project <- list(id=project_id, report=report, datasets=datasets)  
  return(project)
}

#### Create a project object by loading an existent report

In [ ]:
p = load_project("P0000001", "proteomics")

#### Visualize the list of plots contained in the report

In [ ]:
names(p$report$plots)

#### Access a specific plot and use plotly to visualize it

In [ ]:
plotly_build(p$report$plots$`2~peptides~barplot_0_figure`)

#### In the case of networks, they have to be converted from a json format, to an edge list. We use R's networkd3 to create a D3 JavaScript force directed network graph from it.

In [ ]:
importGraph(p$report$nets$`0_net`)

#### We can also, easily, access the different datasets from the project

In [ ]:
names(p$datasets$data)

In [ ]:
p$datasets$data$correlation_correlation